<a href="https://colab.research.google.com/github/mu-janane-27/Innomatics-Research-lab-tasks/blob/main/Task%201%20Build%20a%20Robust%20NLP%20Preprocessing%20Engine%20(Advanced).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Task 1: Conceptual Understanding
1) "Love" vs. "love": The Case Sensitivity Conflict
In NLP, computers don't inherently know that "Love" and "love" are the same word. By default, they see them as two distinct unique identifiers because their ASCII/Unicode values are different.

"love" (lowercase): Usually treated as a standard verb or noun.

"Love" (capitalized): Might be recognized as a Proper Noun (like a surname or a brand) or the start of a sentence.

Why it matters: If we don't "normalize" text (usually by lowercase-ing everything), a model might split its "understanding" between the two. It thinks they are different concepts, which weakens the statistical patterns the model is trying to learn.

2) What happens if Stopwords are NOT removed?
Stopwords are those high-frequency words like "the," "is," and "at." If you leave them in:

Noise Overload: Your model might decide that "the" is the most important word in a document simply because it appears 50 times, even though it carries zero unique meaning.

Performance Drag: The "vocabulary" size of your dataset stays massive. This makes training slower and requires more memory.

Skewed Search Results: In basic search engines, keeping stopwords can lead to irrelevant results because the engine is matching "a" and "of" rather than the actual subject matter.

3) When Removing Stopwords Backfires
While removing them is standard practice, it can sometimes be a total "vibe-killer" for data.

Sentiment Analysis: Imagine the phrase "not good." If you remove the stopword "not," the machine just sees "good" and thinks the user is happy. You’ve just flipped the meaning 180 degrees.

Legal or Literary Analysis: In a contract, the difference between "may" and "must" is everything. In literature, "To be or not to be" consists almost entirely of stopwords. Removing them turns one of history's greatest quotes into... nothing.

4) Stemming vs. Lemmatization
Think of these as two different ways to "clean up" words so the computer sees the root meaning.

Stemming: The Chainsaw Approach
Stemming is a bit "hacky." It chops off the ends of words using fixed rules, hoping to hit the root. It’s fast but crude.

Eg: "Caring" becomes "car".

The Problem: It often creates non-words and can accidentally group unrelated things (like "Caring" and "Cars").

Lemmatization: The Scalpel Approach
Lemmatization is smarter. It uses a dictionary (a "morphological analysis") to find the Lemma, which is the actual dictionary base form of a word. It looks at the context (is this a verb or a noun?).

Eg: "Caring" becomes "care".

The Benefit: It’s much more accurate and keeps the language meaningful, though it takes a bit more "brain power" (computation) for the computer to process

Task 2:Build Advanced Preprocessing Function

In [1]:
import re

def preprocess_text(text):
    if not text or not isinstance(text, str):
        return [], ""

    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\S+@\S+", "", text)
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()
    tokens = [word for word in tokens if len(word) > 2 or word in ["no", "not"]]

    clean_sentence = " ".join(tokens)

    return tokens, clean_sentence

Task 3: Stress Testing

In [8]:
sentences = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

for s in sentences:
    tokens, clean = preprocess_text(s)
    print("Original:", s)
    print("Tokens:", tokens)
    print("Cleaned Sentence:", clean)
    print("-" * 50)

Original: Get 100% FREE access now!!!
Tokens: ['get', 'free', 'access', 'now']
Cleaned Sentence: get free access now
--------------------------------------------------
Original: I absolutely looooved this product 😍😍
Tokens: ['absolutely', 'looved', 'this', 'product']
Cleaned Sentence: absolutely looved this product
--------------------------------------------------
Original: Worst service ever... 0/10
Tokens: ['worst', 'service', 'ever']
Cleaned Sentence: worst service ever
--------------------------------------------------
Original: Call me at 9876543210
Tokens: ['call']
Cleaned Sentence: call
--------------------------------------------------
Original: This is THE best course!!!
Tokens: ['this', 'the', 'best', 'course']
Cleaned Sentence: this the best course
--------------------------------------------------
Original: Visit https://openai.com now!
Tokens: ['visit', 'now']
Cleaned Sentence: visit now
--------------------------------------------------
Original: Nooooo this is baaad!!!


Task 4: Token Analytics

In [14]:
def analyze_tokens(tokens):
    total = len(tokens)
    unique = len(set(tokens))
    avg_len = sum(len(t) for t in tokens) / total if total > 0 else 0
    return total, unique, avg_len

for s in sentences:
    tokens, _ = preprocess_text(s)
    total, unique, avg = analyze_tokens(tokens)

    print("Sentence:", s)
    print("Total:", total, "| Unique:", unique, "| Avg Length:", round(avg, 2))
    print("-" * 50)

Sentence: Get 100% FREE access now!!!
Total: 4 | Unique: 4 | Avg Length: 4.0
--------------------------------------------------
Sentence: I absolutely looooved this product 😍😍
Total: 4 | Unique: 4 | Avg Length: 6.75
--------------------------------------------------
Sentence: Worst service ever... 0/10
Total: 3 | Unique: 3 | Avg Length: 5.33
--------------------------------------------------
Sentence: Call me at 9876543210
Total: 1 | Unique: 1 | Avg Length: 4.0
--------------------------------------------------
Sentence: This is THE best course!!!
Total: 4 | Unique: 4 | Avg Length: 4.25
--------------------------------------------------
Sentence: Visit https://openai.com now!
Total: 2 | Unique: 2 | Avg Length: 4.0
--------------------------------------------------
Sentence: Nooooo this is baaad!!!
Total: 3 | Unique: 3 | Avg Length: 3.67
--------------------------------------------------
Sentence: OK OK OK I got it
Total: 1 | Unique: 1 | Avg Length: 3.0
---------------------------------

Task 5: Frequency Analysis

In [23]:
from collections import Counter

all_tokens = []

for s in sentences:
    tokens, _ = preprocess_text(s)
    all_tokens.extend(tokens)

counter = Counter(all_tokens)

print("Top 10 words:", counter.most_common(10))
print("Least 5 words:", counter.most_common()[:-6:-1])

Top 10 words: [('this', 4), ('now', 3), ('get', 1), ('free', 1), ('access', 1), ('absolutely', 1), ('looved', 1), ('product', 1), ('worst', 1), ('service', 1)]
Least 5 words: [('with', 1), ('happy', 1), ('not', 1), ('offer', 1), ('limited', 1)]


Task 6: Build Full Pipeline

In [29]:
def full_pipeline(text_list):
    all_tokens = []
    clean_sentences = []

    for text in text_list:
        tokens, clean_text = preprocess_text(text)
        all_tokens.extend(tokens)
        clean_sentences.append(clean_text)

    return {
        "tokens": all_tokens,
        "clean_sentences": clean_sentences
    }

result = full_pipeline(sentences)
print(result)

{'tokens': ['get', 'free', 'access', 'now', 'absolutely', 'looved', 'this', 'product', 'worst', 'service', 'ever', 'call', 'this', 'the', 'best', 'course', 'visit', 'now', 'noo', 'this', 'baad', 'got', 'win', 'now', 'limited', 'offer', 'not', 'happy', 'with', 'this'], 'clean_sentences': ['get free access now', 'absolutely looved this product', 'worst service ever', 'call', 'this the best course', 'visit now', 'noo this baad', 'got', 'win now limited offer', 'not happy with this']}


Task 7: Error Handling

The function handles:
- Empty string → returns empty output
- Only emojis → removed during preprocessing
- Only numbers → removed using regex